# Ensemble LEarning- Stacking(Stacked Generalization) with Wine Quality Dataset

 ### Predicting Wine Quality USing Multiple Base Models & Meta Model

In [32]:
#Import required Libraries

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings("ignore")

In [33]:
# Load the Dataset(Direct from UCI)

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep = ';') # ';' is the delimiter

In [34]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [35]:
df['quality'].value_counts()

quality
5    681
6    638
7    199
4     53
8     18
3     10
Name: count, dtype: int64

In [36]:
#Convert the binary Classification Problem
df['quality_binary'] = df['quality'].apply(lambda x: 1 if x >=6 else 0)
df['quality_binary']

0       0
1       0
2       0
3       1
4       0
       ..
1594    0
1595    1
1596    1
1597    0
1598    1
Name: quality_binary, Length: 1599, dtype: int64

In [37]:
# Split the Dataset into Features and Target
X = df.drop(['quality','quality_binary'], axis =1)
y = df['quality_binary']

In [38]:
#Step6:Train-test Split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size =0.2, random_state =42, stratify =y)

In [39]:
X_train.shape,y_train.shape

((1279, 11), (1279,))

In [40]:
X_test.shape,y_test.shape

((320, 11), (320,))

In [41]:
#Feature Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [42]:
base_learners = [
    ('dt', DecisionTreeClassifier(random_state =42)),
    ('knn', KNeighborsClassifier()),
    ('svm',SVC(probability = True,random_state =42))
]

In [43]:
# Define meta-model
meta_model = LogisticRegression()

In [44]:
# Initializing Stacking Model

stack = StackingClassifier(
    estimators = base_learners,
    final_estimator = meta_model,
    cv =5
)

In [45]:
#Train the Stacking Model
stack.fit(X_train_scaled,y_train)

,estimators,"[('dt', ...), ('knn', ...), ...]"
,final_estimator,LogisticRegression()
,cv,5
,stack_method,'auto'
,n_jobs,None
,passthrough,False
,verbose,0
,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2


In [46]:
y_pred = stack.predict(X_test_scaled)

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Accuracy Score:", accuracy_score(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

           0       0.73      0.77      0.75       149
           1       0.79      0.75      0.77       171

    accuracy                           0.76       320
   macro avg       0.76      0.76      0.76       320
weighted avg       0.76      0.76      0.76       320

Accuracy Score: 0.75625
